# Vocab Duel — запуск в Google Colab

Этот ноутбук запускает игру словаря из файлов рядом с ним.

Поддерживается:

- **Одиночный режим** — работает сразу
- **Сетевой режим** — работает в гостевом режиме сразу
- **Google-вход** — включается после загрузки Firebase credentials

In [ ]:
!pip -q install fastapi uvicorn pandas firebase-admin

## 1. Найти папку проекта

Если вы загрузили zip и распаковали его в `/content/vocab_duel_game`, всё определится автоматически.

In [ ]:
import os
from pathlib import Path

CANDIDATES = [
    "/content/vocab_duel_game",
    "/content",
    os.getcwd(),
]
APP_DIR = None
for c in CANDIDATES:
    if Path(c, "backend.py").exists():
        APP_DIR = c
        break

if APP_DIR is None:
    raise FileNotFoundError("Не найден backend.py. Загрузите папку vocab_duel_game в Colab.")

print("APP_DIR =", APP_DIR)
print("Файлы:", os.listdir(APP_DIR)[:20])

## 2. Необязательно: загрузить Firebase service account для Google-входа

Эту ячейку можно **пропустить**, тогда будет гостевой режим.

In [ ]:
USE_FIREBASE = False  # Поставьте True, если хотите включить Google-авторизацию

if USE_FIREBASE:
    from google.colab import files
    uploaded = files.upload()
    print("Загружено:", list(uploaded.keys()))
else:
    print("Firebase отключён. Будет гостевой режим.")

## 3. Необязательно: заполнить публичные Firebase-параметры

Нужно только если выше `USE_FIREBASE = True`.

In [ ]:
import os
from pathlib import Path

# Укажите имя загруженного JSON, если используете Firebase
FIREBASE_JSON_NAME = ""  # например: "service-account.json"

# Публичные ключи Firebase Web App
FIREBASE_API_KEY = ""
FIREBASE_AUTH_DOMAIN = ""
FIREBASE_PROJECT_ID = ""
FIREBASE_STORAGE_BUCKET = ""
FIREBASE_MESSAGING_SENDER_ID = ""
FIREBASE_APP_ID = ""

if USE_FIREBASE:
    if not FIREBASE_JSON_NAME:
        raise ValueError("Укажите FIREBASE_JSON_NAME")
    json_path = Path("/content") / FIREBASE_JSON_NAME
    if not json_path.exists():
        raise FileNotFoundError(f"Не найден {json_path}")

    os.environ["FIREBASE_SERVICE_ACCOUNT"] = str(json_path)
    os.environ["FIREBASE_API_KEY"] = FIREBASE_API_KEY
    os.environ["FIREBASE_AUTH_DOMAIN"] = FIREBASE_AUTH_DOMAIN
    os.environ["FIREBASE_PROJECT_ID"] = FIREBASE_PROJECT_ID
    os.environ["FIREBASE_STORAGE_BUCKET"] = FIREBASE_STORAGE_BUCKET
    os.environ["FIREBASE_MESSAGING_SENDER_ID"] = FIREBASE_MESSAGING_SENDER_ID
    os.environ["FIREBASE_APP_ID"] = FIREBASE_APP_ID

    print("Firebase env подготовлены.")
else:
    print("Пропущено.")

## 4. Запуск backend и открытие игры

In [ ]:
import os, sys, time, subprocess, signal

# Остановить прошлый сервер, если был
try:
    server_proc.terminate()
    time.sleep(1)
except:
    pass

server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=APP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(4)

# Быстрая проверка
import urllib.request
print(urllib.request.urlopen("http://127.0.0.1:8000/api/health").read().decode("utf-8"))

from google.colab import output
output.serve_kernel_port_as_window(8000, path="/")

## 5. Как подать свою Google-таблицу

В интерфейсе игры вставьте ссылку на Google Sheets.

Поддерживаются такие варианты:

- обычная ссылка вида `https://docs.google.com/spreadsheets/d/.../edit#gid=0`
- прямая CSV-ссылка
- пустое поле — тогда используется demo-файл `sample_words.csv`

## 6. Остановка сервера

In [ ]:
try:
    server_proc.terminate()
    print("Сервер остановлен.")
except Exception as e:
    print("Не удалось остановить:", e)